In [0]:
from pyspark.sql.functions import col, upper, row_number
from pyspark.sql.window import Window

print("Iniciando a Camada Silver...")

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

df_customers = spark.table("workspace.bronze.tb_customers")

window_customers = Window.partitionBy("customer_id").orderBy(col("timestamp_ingestion").desc())
df_dim_consumidores = df_customers.withColumn("rn", row_number().over(window_customers)).filter(col("rn") == 1)

cols_consumidores = [
    col("customer_id").alias("id_consumidor"),
    col("customer_zip_code_prefix").alias("prefixo_cep"),
    upper(col("customer_city")).alias("cidade"),
    upper(col("customer_state")).alias("estado")
]

if "customer_name" in df_customers.columns:
    cols_consumidores.append(col("customer_name").alias("nome_consumidor"))

df_dim_consumidores.select(*cols_consumidores).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.dim_consumidores")
print("silver.dim_consumidores processada.")

df_sellers = spark.table("workspace.bronze.tb_sellers")

window_sellers = Window.partitionBy("seller_id").orderBy(col("timestamp_ingestion").desc())
df_dim_vendedores = df_sellers.withColumn("rn", row_number().over(window_sellers)).filter(col("rn") == 1)

cols_vendedores = [
    col("seller_id").alias("id_vendedor"),
    col("seller_zip_code_prefix").alias("prefixo_cep"),
    upper(col("seller_city")).alias("cidade"),
    upper(col("seller_state")).alias("estado")
]
if "seller_name" in df_sellers.columns:
    cols_vendedores.append(col("seller_name").alias("nome_vendedor"))

df_dim_vendedores.select(*cols_vendedores).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.dim_vendedores")
print("silver.dim_vendedores processada.")

df_products = spark.table("workspace.bronze.tb_products")
window_products = Window.partitionBy("product_id").orderBy(col("timestamp_ingestion").desc())
df_dim_produtos = df_products.withColumn("rn", row_number().over(window_products)).filter(col("rn") == 1)

cols_produtos = [
    col("product_id").alias("id_produto"),
    col("product_category_name").alias("categoria_produto"),
    col("product_weight_g").alias("peso_produto_gramas"),
    col("product_length_cm").alias("comprimento_centimetros"),
    col("product_height_cm").alias("altura_centimetros"),
    col("product_width_cm").alias("largura_centimetros"),
    col("product_photos_qty").alias("quantidade_fotos"),
    col("product_description_lenght").alias("tamanho_descricao_produto"),
    col("product_name_lenght").alias("tamanho_nome_produto")
]
if "product_name" in df_products.columns:
    cols_produtos.append(col("product_name").alias("nome_produto"))

df_dim_produtos.select(*cols_produtos).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.dim_produtos")

df_translation = spark.table("workspace.bronze.tb_product_category_name_translation")
df_translation.select(
    col("product_category_name").alias("nome_produto_pt"),
    col("product_category_name_english").alias("nome_produto_en")
).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.dim_categoria_produtos_traducao")
print("silver.dim_produtos e traduções processadas.")

Iniciando a Camada Silver...
✓ silver.dim_consumidores processada.
✓ silver.dim_vendedores processada.
✓ silver.dim_produtos e traduções processadas.


In [0]:
from pyspark.sql.functions import when, datediff, coalesce, lit, current_timestamp, expr

df_orders = spark.table("workspace.bronze.tb_orders")
df_fat_pedidos = df_orders.withColumn("status", 
    when(col("order_status") == "delivered", "entregue")
    .when(col("order_status") == "canceled", "cancelado")
    .when(col("order_status") == "shipped", "enviado")
    .when(col("order_status") == "processing", "processando")
    .when(col("order_status") == "invoiced", "faturado")
    .when(col("order_status") == "unavailable", "indisponível")
    .when(col("order_status") == "created", "criado")
    .when(col("order_status") == "approved", "aprovado")
    .otherwise(col("order_status"))
)

df_fat_pedidos = df_fat_pedidos \
    .withColumn("tempo_entrega_dias", datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))) \
    .withColumn("tempo_entrega_estimado_dias", datediff(col("order_estimated_delivery_date"), col("order_purchase_timestamp"))) \
    .withColumn("diferenca_entrega_dias", datediff(col("order_delivered_customer_date"), col("order_estimated_delivery_date"))) \
    .withColumn("entrega_no_prazo", 
        when(col("status") != "entregue", "Não Entregue")
        .when(col("diferenca_entrega_dias") <= 0, "Sim")
        .otherwise("Não")
    )

df_fat_pedidos.withColumnRenamed("order_id", "id_pedido") \
    .withColumnRenamed("customer_id", "id_consumidor") \
    .withColumnRenamed("order_purchase_timestamp", "data_pedido") \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.fat_pedidos")

spark.table("workspace.bronze.tb_order_items").select(
    col("order_id").alias("id_pedido"), col("order_item_id").alias("id_item"), col("product_id").alias("id_produto"),
    col("seller_id").alias("id_vendedor"), col("price").alias("preco_BRL"), col("freight_value").alias("preco_frete")
).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.fat_itens_pedidos")
print("Fato Pedidos e Itens processadas.")

spark.table("workspace.bronze.tb_order_payments").withColumn("tipo_pagamento",
    when(col("payment_type") == "credit_card", "Cartão de Crédito").when(col("payment_type") == "boleto", "Boleto")
    .when(col("payment_type") == "voucher", "Voucher").when(col("payment_type") == "debit_card", "Cartão de Débito")
    .when(col("payment_type") == "not_defined", "Não Definido").otherwise(col("payment_type"))
).withColumnRenamed("order_id", "id_pedido").withColumnRenamed("payment_value", "valor_pagamento") \
.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.fat_pagamentos_pedidos")

df_reviews = spark.table("workspace.bronze.tb_order_reviews").select(
    col("review_id").alias("id_avaliacao"), col("order_id").alias("id_pedido"), col("review_score").alias("nota_avaliacao"),
    coalesce(col("review_comment_title"), lit("Sem título")).alias("titulo_avaliacao"),
    coalesce(col("review_comment_message"), lit("Sem comentário")).alias("comentario_avaliacao"),
    expr("try_to_timestamp(review_creation_date)").alias("data_criacao_avaliacao"),
    expr("try_to_timestamp(review_answer_timestamp)").alias("data_resposta_avaliacao")
)
df_reviews.filter(col("id_pedido").isNotNull()).filter(col("data_criacao_avaliacao") <= current_timestamp()) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.fat_avaliacoes_pedidos")
print("Fato Pagamentos e Avaliações processadas.")

Fato Pedidos e Itens processadas.
Fato Pagamentos e Avaliações processadas.


In [0]:
from pyspark.sql.functions import to_date, last, sum, round

df_dolar = spark.table("workspace.bronze.tb_cotacao_dolar").withColumn("data_cotacao", to_date(col("dataHoraCotacao")))

min_date = df_dolar.agg({"data_cotacao": "min"}).collect()[0][0]
max_date = df_dolar.agg({"data_cotacao": "max"}).collect()[0][0]

df_calendario = spark.sql(f"SELECT explode(sequence(to_date('{min_date}'), to_date('{max_date}'), interval 1 day)) as data_calendario")
df_dolar_completo = df_calendario.join(df_dolar, df_calendario.data_calendario == df_dolar.data_cotacao, "left")

window_dolar = Window.orderBy("data_calendario").rowsBetween(Window.unboundedPreceding, Window.currentRow)
df_dim_cotacao = df_dolar_completo.withColumn("cotacao_compra", last(col("cotacaoCompra"), ignorenulls=True).over(window_dolar)) \
    .select(col("data_calendario").alias("data_cotacao"), "cotacao_compra")

df_dim_cotacao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.dim_cotacao_dolar")

df_pedidos = spark.table("workspace.silver.fat_pedidos")
df_pagamentos = spark.table("workspace.silver.fat_pagamentos_pedidos").groupBy("id_pedido").agg(sum("valor_pagamento").alias("valor_total_pago_brl"))

df_final = df_pedidos.join(df_pagamentos, "id_pedido", "left").withColumn("data_pedido_join", to_date(col("data_pedido")))
df_final = df_final.join(spark.table("workspace.silver.dim_cotacao_dolar"), df_final.data_pedido_join == col("data_cotacao"), "left")

df_final = df_final.withColumn("valor_total_pago_usd", round(col("valor_total_pago_brl") / col("cotacao_compra"), 2)) \
                   .withColumn("valor_total_pago_brl", round(col("valor_total_pago_brl"), 2))

df_final.select("id_pedido", "id_consumidor", "status", "valor_total_pago_brl", "valor_total_pago_usd", "data_pedido") \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.fat_pedido_total")

print("Tabelas geradas. Executando Otimização com ZORDER")
spark.sql("OPTIMIZE workspace.silver.fat_pedido_total ZORDER BY (id_pedido, data_pedido)")
print("Camada Silver concluída.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Tabelas geradas. Executando Otimização com ZORDER
Camada Silver concluída.
